***I will train 2 Logistic Regression models where Model 2 accounts for the moment in time that the pass is made during the match, and Model 1 doesn't***

In [1]:
import pandas as pd

df = pd.read_csv('../data/processed/processed_passes.csv')

df.columns

Index(['match_id', 'period', 'half_percentage', 'net_score', 'under_pressure',
       'start_x', 'start_y', 'end_x', 'end_y', 'pass_angle', 'pass_length',
       'dist_to_goal', 'dist_receiv_middle', 'is_progressive', 'pass_outcome'],
      dtype='str')

In [67]:
features1 = [ 
    'under_pressure',
    'start_x', 
    'start_y', 
    'end_x', 
    'end_y', 
    'pass_angle', 
    'pass_length',
    'dist_to_goal', 
    'dist_receiv_middle'
]

In [68]:
features2 = [
    'period', 
    'half_percentage', 
    'net_score', 
    'under_pressure',
    'start_x', 
    'start_y', 
    'end_x', 
    'end_y', 
    'pass_angle', 
    'pass_length',
    'dist_to_goal', 
    'dist_receiv_middle'
]

In [69]:
features3 = [ 
    'under_pressure',
    'start_x', 
    'start_y', 
    'end_x', 
    'end_y', 
    'pass_angle', 
    'pass_length',
    'dist_to_goal', 
    'dist_receiv_middle',
    'is_progressive'
]

In [70]:
def train_xgb(features):

    print('Splitting the data...')

    X = df[features]
    y = df['pass_outcome']

    from sklearn.model_selection import train_test_split
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42)

    print(f"Training Passes: {len(X_train)} passes")
    print(f"Testing Passes: {len(X_test)} passes")

    import xgboost as xgb

    # Specify the parameters of my models
    xgb_config = {
        'n_estimators': 100,
        'max_depth': 4,
        'learning_rate': 0.05,
        'subsample': 0.8,
        'colsample_bytree': 0.8,
        'random_state': 42,
        'eval_metric': 'logloss'
    }

    xgb1 = xgb.XGBClassifier(**xgb_config)
    




In [71]:
X1 = df[features1]
y = df['pass_outcome']

In [72]:
X2 = df[features2]
X3 = df[features3]

In [73]:
# Divide my data to training and test with an 80% to 20% split respectively
from sklearn.model_selection import train_test_split
X_train1, X_test1, y_train1, y_test1 = train_test_split(X1, y, test_size = 0.2, random_state = 42)

In [74]:
X_train2, X_test2, y_train2, y_test2 = train_test_split(X2, y, test_size = 0.2, random_state = 42)
X_train3, X_test3, y_train3, y_test3 = train_test_split(X3, y, test_size = 0.2, random_state = 42)

In [75]:
print(f"Training Passes: {len(X_train1)} passes")
print(f"Testing Passes: {len(X_test1)} passes")

Training Passes: 32269 passes
Testing Passes: 8068 passes


In [76]:
# Prevent jupyter notebook html display errors
import sklearn
sklearn.set_config(display="text")

In [77]:
# Firstly I will train a Logistic Regression model

from sklearn.linear_model import LogisticRegression

log_reg_model1 = LogisticRegression(max_iter=1000)
log_reg_model1.fit(X_train1, y_train1)

LogisticRegression(max_iter=1000)

In [78]:
from sklearn.linear_model import LogisticRegression

log_reg_model2 = LogisticRegression(max_iter=1000)
log_reg_model2.fit(X_train2, y_train2)

LogisticRegression(max_iter=1000)

In [79]:
from sklearn.metrics import accuracy_score

log_reg_train_predictions1 = log_reg_model1.predict(X_train1)
log_reg_test_predictions1 = log_reg_model1.predict(X_test1)

print(f"Training Accuracy for Model 1: {accuracy_score(y_train1, log_reg_train_predictions1)*100:.4f} %")
print(f"Testing Accuracy for Model 1: {accuracy_score(y_test1, log_reg_test_predictions1)*100:.4f} %")

Training Accuracy for Model 1: 86.8233 %
Testing Accuracy for Model 1: 86.5394 %


In [80]:
from sklearn.metrics import accuracy_score

log_reg_train_predictions2 = log_reg_model2.predict(X_train2)
log_reg_test_predictions2 = log_reg_model2.predict(X_test2)

print(f"Training Accuracy for Model 2: {accuracy_score(y_train2, log_reg_train_predictions2)*100:.4f} %")
print(f"Testing Accuracy for Model 2: {accuracy_score(y_test2, log_reg_test_predictions2)*100:.4f} %")

Training Accuracy for Model 2: 86.8388 %
Testing Accuracy for Model 2: 86.5518 %


In [81]:
#Calculate Probabilistic Metrics for Model 1

from sklearn.metrics import log_loss, brier_score_loss, roc_auc_score

log_reg_y_prob1 = log_reg_model1.predict_proba(X_test1)[:, 1]

log_reg_baseline_log_loss1 = log_loss(y_test1, log_reg_y_prob1)
log_reg_baseline_brier1 = brier_score_loss(y_test1, log_reg_y_prob1)
log_reg_baseline_auc1 = roc_auc_score(y_test1, log_reg_y_prob1)

print(f"Baseline Log-Loss for Model 1: {log_reg_baseline_log_loss1:.5f}")
print(f"Baseline Brier Score for Model 1: {log_reg_baseline_brier1:.5f}")
print(f"Baseline ROC-AUC for Model 1: {log_reg_baseline_auc1:.5f}")

Baseline Log-Loss for Model 1: 0.33189
Baseline Brier Score for Model 1: 0.10045
Baseline ROC-AUC for Model 1: 0.79985


In [82]:
#Calculate Probabilistic Metrics for Model 2

from sklearn.metrics import log_loss, brier_score_loss, roc_auc_score

log_reg_y_prob2 = log_reg_model2.predict_proba(X_test2)[:, 1]

log_reg_baseline_log_loss2 = log_loss(y_test2, log_reg_y_prob2)
log_reg_baseline_brier2 = brier_score_loss(y_test2, log_reg_y_prob2)
log_reg_baseline_auc2 = roc_auc_score(y_test2, log_reg_y_prob2)

print(f"Baseline Log-Loss for Model 2: {log_reg_baseline_log_loss2:.5f}")
print(f"Baseline Brier Score for Model 2: {log_reg_baseline_brier2:.5f}")
print(f"Baseline ROC-AUC for Model 2: {log_reg_baseline_auc2:.5f}")

Baseline Log-Loss for Model 2: 0.33179
Baseline Brier Score for Model 2: 0.10049
Baseline ROC-AUC for Model 2: 0.80020


***I will now train and compare in a similar way Models with different features using XGBOOST***

In [6]:
def train_xgb(features):

    print('Splitting the data...')

    X = df[features]
    y = df['pass_outcome']

    from sklearn.model_selection import train_test_split
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42)

    print(f"Training Passes: {len(X_train)} passes")
    print(f"Testing Passes: {len(X_test)} passes")

    print('\nTraining XGBOOST Model ...')
    import xgboost as xgb

    # Specify the parameters of my models
    xgb_config = {
        'n_estimators': 100,
        'max_depth': 4,
        'learning_rate': 0.05,
        'subsample': 0.8,
        'colsample_bytree': 0.8,
        'random_state': 42,
        'eval_metric': 'logloss'
    }

    xgb1 = xgb.XGBClassifier(**xgb_config)


    xgb1.fit(X_train, y_train)
    print('XGBOOST Model training Done!')

    from sklearn.metrics import accuracy_score
    xgb_train_predictions = xgb1.predict(X_train)
    xgb_test_predictions = xgb1.predict(X_test)
    print(f"\nTraining Accuracy for XGBOOST Model: {accuracy_score(y_train, xgb_train_predictions)*100:.4f} %")
    print(f"Testing Accuracy for XGBOOST Model: {accuracy_score(y_test, xgb_test_predictions)*100:.4f} %")  

    #Calculate Probabilistic Metrics for XGBOOST Model (Basic Metrics)
    
    from sklearn.metrics import log_loss, brier_score_loss, roc_auc_score
    xgb_y_prob = xgb1.predict_proba(X_test)[:, 1]

    xgb_baseline_log_loss = log_loss(y_test, xgb_y_prob)
    xgb_baseline_brier = brier_score_loss(y_test, xgb_y_prob)
    xgb_baseline_auc = roc_auc_score(y_test, xgb_y_prob)

    print(f"\nBaseline Log-Loss for XGBOOST Model: {xgb_baseline_log_loss:.5f}")
    print(f"Baseline Brier Score for XGBOOST Model: {xgb_baseline_brier:.5f}")
    print(f"Baseline ROC-AUC for XGBOOST Model: {xgb_baseline_auc:.5f}")

In [90]:
df.columns

Index(['match_id', 'period', 'half_percentage', 'net_score', 'under_pressure',
       'start_x', 'start_y', 'end_x', 'end_y', 'pass_angle', 'pass_length',
       'dist_to_goal', 'dist_receiv_middle', 'is_progressive', 'pass_outcome'],
      dtype='str')

In [16]:
feat1 = [
    'period', 'half_percentage', 'net_score', 'under_pressure',
       'start_x', 'start_y', 'end_x', 'end_y', 'pass_angle', 'pass_length',
       'dist_to_goal', 'dist_receiv_middle', 'is_progressive'
]

print('---SPATIAL + TEMPORAL/SCORELINE + PROGRESSIVE---\n')
train_xgb(feat1)

---SPATIAL + TEMPORAL/SCORELINE + PROGRESSIVE---

Splitting the data...
Training Passes: 32269 passes
Testing Passes: 8068 passes

Training XGBOOST Model ...
XGBOOST Model training Done!

Training Accuracy for XGBOOST Model: 90.4397 %
Testing Accuracy for XGBOOST Model: 89.7744 %

Baseline Log-Loss for XGBOOST Model: 0.26638
Baseline Brier Score for XGBOOST Model: 0.07835
Baseline ROC-AUC for XGBOOST Model: 0.88418


In [18]:
feat2 = [
    'period', 'half_percentage', 'net_score', 'under_pressure',
       'start_x', 'start_y', 'end_x', 'end_y', 'pass_angle', 'pass_length',
       'dist_to_goal', 'dist_receiv_middle'
]

print('---SPATIAL + TEMPORAL/SCORELINE---\n')
train_xgb(feat2)

---SPATIAL + TEMPORAL/SCORELINE---

Splitting the data...
Training Passes: 32269 passes
Testing Passes: 8068 passes

Training XGBOOST Model ...
XGBOOST Model training Done!

Training Accuracy for XGBOOST Model: 90.4397 %
Testing Accuracy for XGBOOST Model: 89.7992 %

Baseline Log-Loss for XGBOOST Model: 0.26720
Baseline Brier Score for XGBOOST Model: 0.07855
Baseline ROC-AUC for XGBOOST Model: 0.88277


In [19]:
feat3 = [
    'under_pressure',
       'start_x', 'start_y', 'end_x', 'end_y', 'pass_angle', 'pass_length',
       'dist_to_goal', 'dist_receiv_middle', 'is_progressive'
]

print('---SPATIAL + PROGRESSIVE---\n')
train_xgb(feat3)

---SPATIAL + PROGRESSIVE---

Splitting the data...
Training Passes: 32269 passes
Testing Passes: 8068 passes

Training XGBOOST Model ...
XGBOOST Model training Done!

Training Accuracy for XGBOOST Model: 90.4800 %
Testing Accuracy for XGBOOST Model: 89.7868 %

Baseline Log-Loss for XGBOOST Model: 0.26657
Baseline Brier Score for XGBOOST Model: 0.07839
Baseline ROC-AUC for XGBOOST Model: 0.88347


In [20]:
feat4 = [
    'under_pressure',
       'start_x', 'start_y', 'end_x', 'end_y', 'pass_angle', 'pass_length',
       'dist_to_goal', 'dist_receiv_middle'
]

print('---SPATIAL---\n')
train_xgb(feat4)

---SPATIAL---

Splitting the data...
Training Passes: 32269 passes
Testing Passes: 8068 passes

Training XGBOOST Model ...
XGBOOST Model training Done!

Training Accuracy for XGBOOST Model: 90.4738 %
Testing Accuracy for XGBOOST Model: 89.8240 %

Baseline Log-Loss for XGBOOST Model: 0.26588
Baseline Brier Score for XGBOOST Model: 0.07819
Baseline ROC-AUC for XGBOOST Model: 0.88440


In [23]:
feat5 = [
       'start_x', 'start_y', 'end_x', 'end_y', 'pass_angle', 'pass_length',
       'dist_to_goal', 'dist_receiv_middle'
]

print('---SPATIAL EXCLUDING UNDER PRESSURE---\n')
train_xgb(feat5)

---SPATIAL EXCLUDING UNDER PRESSURE---

Splitting the data...
Training Passes: 32269 passes
Testing Passes: 8068 passes

Training XGBOOST Model ...
XGBOOST Model training Done!

Training Accuracy for XGBOOST Model: 90.3809 %
Testing Accuracy for XGBOOST Model: 89.8364 %

Baseline Log-Loss for XGBOOST Model: 0.26988
Baseline Brier Score for XGBOOST Model: 0.07910
Baseline ROC-AUC for XGBOOST Model: 0.87665


In [25]:
feat6 = [
       'start_x', 'start_y', 'end_x', 'end_y', 'pass_angle', 'pass_length',
       'dist_to_goal', 'dist_receiv_middle', 'is_progressive'
]

print('---SPATIAL + PROGRESSIVE EXCLUDING UNDER PRESSURE---\n')
train_xgb(feat6)

---SPATIAL + PROGRESSIVE EXCLUDING UNDER PRESSURE---

Splitting the data...
Training Passes: 32269 passes
Testing Passes: 8068 passes

Training XGBOOST Model ...
XGBOOST Model training Done!

Training Accuracy for XGBOOST Model: 90.3995 %
Testing Accuracy for XGBOOST Model: 89.7124 %

Baseline Log-Loss for XGBOOST Model: 0.26981
Baseline Brier Score for XGBOOST Model: 0.07910
Baseline ROC-AUC for XGBOOST Model: 0.87668


***I realise that the pure spatial metrics, combined with the 'under_pressure' flag, result in the best performing model***